# Semi-automated STGT vs Adaptive Fusion GAT-LSTM

This notebook is intentionally gated. It uses the approved workflow only: WSL/GitHub for source, Google Drive for data/artifacts, and standard Google Colab for compute. It does not use a Colab API, MCP server, browser automation, or unattended authorization.

**Training does not start unless `RUN_EXPERIMENTS` is explicitly changed to `True` after all preflight checks pass.**

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPO_URL = 'https://github.com/Akifali-1/research.git'
REPO_BRANCH = 'research/initial-pipeline'
PROJECT = Path('/content/research')
RUN_EXPERIMENTS = False  # change manually only after every gate passes
MAX_GPU_RUNS = 10
if PROJECT.exists():
    raise RuntimeError(f'{PROJECT} already exists; inspect it before continuing')
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT)], check=True)
%cd /content/research
local_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
remote_line = subprocess.check_output(['git', 'ls-remote', 'origin', f'refs/heads/{REPO_BRANCH}'], text=True).strip()
if not remote_line or remote_line.split()[0] != local_commit:
    raise RuntimeError('Repository push gate failed: local HEAD does not match origin/HEAD')
print('Repository commit:', local_commit)

## Interactive Google Drive authorization

Run this cell interactively. The notebook cannot continue until Google Drive is mounted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/supply_chain_research')
RAW_DIR = DRIVE_ROOT / 'data' / 'raw'
PROCESSED_DIR = DRIVE_ROOT / 'data' / 'processed'
ARTIFACT_ROOT = DRIVE_ROOT / 'experiments'
REGISTRY_PATH = DRIVE_ROOT / 'experiment_registry.json'
for path in [RAW_DIR, PROCESSED_DIR, ARTIFACT_ROOT]: path.mkdir(parents=True, exist_ok=True)
print('Drive authorized at:', DRIVE_ROOT)

In [ ]:
%pip install -q -r requirements.txt
import torch, numpy, pandas, sklearn, yaml
print('torch:', torch.__version__)
print('numpy:', numpy.__version__)
print('pandas:', pandas.__version__)
print('sklearn:', sklearn.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    print('No GPU; training remains disabled unless explicitly chosen')

In [ ]:
sys.path.insert(0, str(PROJECT))
from orchestrator import validate_archives
from src.experiment_registry import Registry
manifest = validate_archives(RAW_DIR)
print('Validated archives:', len(manifest['archives']))
registry = Registry(REGISTRY_PATH, max_runs=MAX_GPU_RUNS, max_retries=2)
interrupted = registry.mark_interrupted_active('new interactive Colab session')
print('Marked stale active records interrupted:', interrupted)
print('Budget:', registry.status())
if registry.status()['remaining_runs'] <= 0:
    raise RuntimeError('Experiment budget is exhausted; no GPU run may be submitted')

## Gate: do not start compute before explicit confirmation

The previous cell verifies the pushed commit, interactive Drive authorization, all Favorita archives, and remaining budget. Review its output. The next cell requires a separate explicit confirmation; do not type `RUN` until those checks are acceptable.

In [ ]:
confirmation = input('Type RUN only after reviewing repository, Drive, archive, and budget checks: ').strip()
if confirmation != 'RUN':
    raise RuntimeError('Safety gate not confirmed; no experiment was submitted')
print('Explicit execution confirmation received')

In [ ]:
import yaml
config = yaml.safe_load((PROJECT / 'configs' / 'experiment.yaml').read_text())
config['data']['processed_dir'] = str(PROCESSED_DIR)
config['outputs']['checkpoint_dir'] = str(ARTIFACT_ROOT / 'checkpoints')
config['outputs']['predictions_dir'] = str(ARTIFACT_ROOT / 'predictions')
config['outputs']['metrics_dir'] = str(ARTIFACT_ROOT / 'metrics')
config['outputs']['logs_dir'] = str(ARTIFACT_ROOT / 'logs')
config['outputs']['drive_artifact_root'] = str(ARTIFACT_ROOT)
COLAB_CONFIG = Path('/content/experiment_colab.yaml')
COLAB_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print(COLAB_CONFIG.read_text())

In [ ]:
command = [sys.executable, 'orchestrator.py', '--project-root', str(PROJECT), '--branch', REPO_BRANCH, '--config', str(COLAB_CONFIG), '--raw-dir', str(RAW_DIR), '--registry', str(REGISTRY_PATH), '--models', 'stgt', 'gat_lstm', '--max-runs', str(MAX_GPU_RUNS), '--allow-execution']
print('Sequential controller command:', ' '.join(command))
subprocess.run(command, cwd=PROJECT, check=True)

In [ ]:
print(json.dumps(registry.status(), indent=2))
print('Experiment registry:', REGISTRY_PATH)